# 第11讲：数据读取与写入

## 课程简介

在上一讲中，我们学习了 Pandas 的基础数据结构和常用操作。  
本讲将进一步学习如何把外部数据读入 Pandas，以及如何把处理后的数据保存到文件中。

真实的数据分析任务通常从“读取数据”开始。数据可能来自 CSV、TXT、Excel，也可能来自更复杂的日志文件或数据库导出文件。  
因此，熟练掌握数据读取与写入，是进行后续数据清洗和分析的前提。

本讲重点包括：

- 读取 CSV 和文本文件
- 处理没有表头、特殊分隔符和缺失值标记的文件
- 逐块读取大文件
- 将 DataFrame 写出为 CSV
- 使用 Python 内置 `csv` 模块处理分隔符文件
- 读取和写入 Excel 文件


## 1 导入库与准备示例数据

<div class="alert alert-success">

**环境提示**

本讲主要使用 Pandas。若需要读写 Excel 文件，还需要安装 `openpyxl`：

```bash
pip install pandas openpyxl
```

如果使用 Anaconda，通常已经安装了这些库。

</div>


为了保证课件可以独立运行，本讲会先自动创建一组示例文件。  
这些文件会保存在当前工作目录下的 `examples/` 文件夹中，后面的代码会直接读取它们。


In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
import sys
import csv

np.random.seed(12345)

pd.options.display.max_rows = 20
pd.options.display.max_columns = 20
pd.options.display.max_colwidth = 80
np.set_printoptions(precision=4, suppress=True)

example_dir = Path("examples")
example_dir.mkdir(exist_ok=True)

# 1. 常规 CSV：带表头
(example_dir / "ex1.csv").write_text(
    "a,b,c,d,message\n"
    "1,2,3,4,hello\n"
    "5,6,7,8,world\n"
    "9,10,11,12,foo\n",
    encoding="utf-8"
)

# 2. 无表头 CSV
(example_dir / "ex2.csv").write_text(
    "1,2,3,4,hello\n"
    "5,6,7,8,world\n"
    "9,10,11,12,foo\n",
    encoding="utf-8"
)

# 3. 多层索引 CSV
(example_dir / "csv_mindex.csv").write_text(
    "key1,key2,value1,value2\n"
    "one,a,1,2\n"
    "one,b,3,4\n"
    "one,c,5,6\n"
    "two,a,7,8\n"
    "two,b,9,10\n"
    "two,c,11,12\n",
    encoding="utf-8"
)

# 4. 使用不规则空白符分隔的文本
(example_dir / "ex3.txt").write_text(
    "            A         B         C\n"
    "aaa -0.264438 -1.026059 -0.619500\n"
    "bbb  0.927272  0.302904 -0.032399\n"
    "ccc -0.264273 -0.386314 -0.217601\n"
    "ddd -0.871858 -0.348382  1.100491\n",
    encoding="utf-8"
)

# 5. 需要跳过行的 CSV
(example_dir / "ex4.csv").write_text(
    "# hey!\n"
    "a,b,c,d,message\n"
    "# just wanted to make things more difficult for you\n"
    "# who reads CSV files with computers, anyway?\n"
    "1,2,3,4,hello\n"
    "5,6,7,8,world\n"
    "9,10,11,12,foo\n",
    encoding="utf-8"
)

# 6. 包含缺失值标记的 CSV
(example_dir / "ex5.csv").write_text(
    "something,a,b,c,d,message\n"
    "one,1,2,3,4,NA\n"
    "two,5,6,,8,world\n"
    "three,9,10,11,12,foo\n"
    "four,13,14,15,16,NULL\n",
    encoding="utf-8"
)

# 7. 较大的 CSV，用于逐块读取
keys = np.random.choice(list("abcd"), size=10000)
values = np.random.standard_normal(10000)
pd.DataFrame({"key": keys, "value": values}).to_csv(example_dir / "ex6.csv", index=False)

# 8. 带引号和逗号的 CSV
(example_dir / "ex7.csv").write_text(
    '"a","b","c"\n'
    '"1","2","3"\n'
    '"1","2","3"\n',
    encoding="utf-8"
)

# 9. Excel 示例文件
excel_frame = pd.DataFrame({
    "a": [1, 5, 9],
    "b": [2, 6, 10],
    "c": [3, 7, 11],
    "d": [4, 8, 12],
    "message": ["hello", "world", "foo"]
})
excel_frame.to_excel(example_dir / "ex1.xlsx", sheet_name="Sheet1", index=False)

def show_text_file(path):
    """显示文本文件内容"""
    print(Path(path).read_text(encoding="utf-8"))

print("示例数据已准备完成。")
print("示例文件夹：", example_dir.resolve())


## 2 读写文本格式的数据

Pandas 提供了多种函数，可以将表格型数据读取为 DataFrame。  
其中最常用的是：

| 函数 | 常见用途 |
|---|---|
| `pd.read_csv()` | 读取 CSV 或其他分隔符文本文件 |
| `pd.read_table()` | 读取表格型文本文件，默认分隔符是制表符 |
| `pd.read_excel()` | 读取 Excel 文件 |
| `pd.read_json()` | 读取 JSON 数据 |
| `pd.read_html()` | 从网页表格中读取数据 |

在读取文件时，常见参数可以分为几类：

- **索引与表头**：是否从文件中读取列名，是否把某些列作为索引
- **分隔符**：逗号、制表符、空白符或自定义分隔符
- **缺失值处理**：识别哪些字符串代表缺失值
- **数据类型推断**：自动判断每一列的数据类型
- **分块读取**：面对大文件时逐块处理
- **异常格式处理**：跳过说明行、注释行或不规则行

下面先从一个最常见的 CSV 文件开始。


In [ ]:
# 查看示例文本文件的原始内容
show_text_file("examples/ex1.csv")

由于该文件以逗号分隔，因此可以使用 `pd.read_csv()` 将其读入为 DataFrame。

In [ ]:
df = pd.read_csv("examples/ex1.csv")
df

也可以使用 `pd.read_table()`，并通过 `sep=","` 指定分隔符。  
不过对于 CSV 文件，通常直接使用 `pd.read_csv()` 更直观。


In [ ]:
pd.read_table('examples/ex1.csv', sep=',')

### 2.1 读取没有表头的文件

并不是所有文件都有标题行。下面这个文件没有列名。


In [ ]:
show_text_file("examples/ex2.csv")


读取没有表头的文件时，有两种常见方法：

1. 让 Pandas 自动分配默认列名
2. 使用 `names` 参数手动指定列名


In [ ]:
pd.read_csv("examples/ex2.csv", header=None)

In [ ]:
pd.read_csv("examples/ex2.csv", names=["a", "b", "c", "d", "message"])

如果希望将某一列作为 DataFrame 的索引，可以使用 `index_col` 参数。  
例如，将 `message` 列设置为行索引。


In [ ]:
names = ["a", "b", "c", "d", "message"]
pd.read_csv("examples/ex2.csv", names=names, index_col="message")

### 2.2 读取多层索引文件

如果希望将多个列组成层次化索引，可以给 `index_col` 传入列名列表。


In [ ]:
show_text_file("examples/csv_mindex.csv")


In [ ]:
parsed = pd.read_csv("examples/csv_mindex.csv",
                     index_col=["key1", "key2"])
parsed

### 2.3 读取不规则分隔符文件

有些表格并不是用固定的逗号或制表符分隔，而是用数量不等的空白字符分隔。  
这种情况下，可以使用正则表达式作为分隔符。


In [ ]:
show_text_file("examples/ex3.txt")


这里的字段由数量不等的空白字符分隔。  
正则表达式 `r"\s+"` 表示匹配一个或多个连续空白字符，适合读取这类文本。


In [ ]:
result = pd.read_csv("examples/ex3.txt", sep=r"\s+")
result


### 2.4 跳过无关行

有些文件在正式数据前会包含说明行、注释行或日志信息。  
可以使用 `skiprows` 跳过指定行。


In [ ]:
show_text_file("examples/ex4.csv")


In [ ]:
pd.read_csv("examples/ex4.csv", skiprows=[0, 2, 3])

### 2.5 识别缺失值

缺失值处理是文件解析中的重要任务。  
默认情况下，Pandas 会识别一些常见缺失值标记，例如空字符串、`NA`、`NULL` 等。


In [ ]:
show_text_file("examples/ex5.csv")


In [ ]:
result = pd.read_csv("examples/ex5.csv")
result

In [ ]:
pd.isna(result)

如果数据中使用了特殊标记表示缺失值，可以通过 `na_values` 自定义。

In [ ]:
result = pd.read_csv("examples/ex5.csv", na_values=["NULL"])
result

也可以用字典为不同列指定不同的缺失值标记。

In [ ]:
sentinels = {"message": ["foo", "NA"], "something": ["two"]}
pd.read_csv("examples/ex5.csv", na_values=sentinels,
            keep_default_na=False)

### <font color='darkorange'><b>动手练习 1</b></font>

#### 题目
请读取 `examples/ex5.csv`，并完成以下任务：

1. 将字符串 `"NULL"` 识别为缺失值
2. 查看每一列缺失值的数量
3. 将 `message` 列设置为索引

#### 你的答案
请在下方代码单元中完成练习。


In [ ]:
# Write your code here



#### 参考答案

<details>
<summary>点击查看示例代码</summary>

```python
practice = pd.read_csv("examples/ex5.csv", na_values=["NULL"])
print(practice.isna().sum())

practice = practice.set_index("message")
practice
```

</details>


### `read_csv()` 常用参数

| 参数 | 说明 |
|---|---|
| `sep` 或 `delimiter` | 指定字段分隔符 |
| `header` | 指定哪一行作为列名，`None` 表示没有表头 |
| `names` | 手动指定列名 |
| `index_col` | 指定一列或多列作为索引 |
| `skiprows` | 跳过指定行 |
| `na_values` | 指定额外的缺失值标记 |
| `nrows` | 只读取前 N 行 |
| `chunksize` | 逐块读取，每块包含的行数 |
| `encoding` | 指定文件编码，例如 `"utf-8"` |


### 2.6 逐块读取文本文件

当文件很大时，一次性读入全部数据可能会占用较多内存。  
此时可以先读取少量行进行观察，或者使用 `chunksize` 逐块读取。


In [ ]:
pd.options.display.max_rows = 10

In [ ]:
result = pd.read_csv("examples/ex6.csv")
result

如果只想读取前几行，可以使用 `nrows` 参数。

In [ ]:
pd.read_csv("examples/ex6.csv", nrows=5)

指定 `chunksize` 后，`read_csv()` 会返回一个可迭代对象。  
每次迭代都会读入一块数据。


In [ ]:
chunker = pd.read_csv("examples/ex6.csv", chunksize=1000)
type(chunker)

下面的例子按块读取 `ex6.csv`，并累计统计 `key` 列中每个取值出现的次数。

In [ ]:
chunker = pd.read_csv("examples/ex6.csv", chunksize=1000)

tot = pd.Series([], dtype='int64')
for piece in chunker:
    tot = tot.add(piece["key"].value_counts(), fill_value=0)

tot = tot.sort_values(ascending=False)

In [ ]:
tot

### <font color='cornflowerblue'><b>思考题 1</b></font>

为什么处理大文件时，`chunksize` 比一次性读取整个文件更安全？

#### 参考答案

<details>
<summary>点击查看解释</summary>

一次性读取整个文件会把所有数据同时加载到内存中。  
如果文件很大，可能导致运行速度变慢，甚至内存不足。

使用 `chunksize` 可以逐块读取和处理数据，每次只在内存中保留一小部分数据，更适合日志文件、交易记录等大规模数据。

</details>


### 2.7 将数据写出到文本格式

除了读取文件，Pandas 也可以把 DataFrame 写出为 CSV 或其他分隔符格式。  
最常用的方法是 `DataFrame.to_csv()`。


In [ ]:
data = pd.read_csv("examples/ex5.csv")
data

利用DataFrame的`to_csv()`方法，我们可以将数据写到一个以逗号分隔的文件中：

In [ ]:
data.to_csv("examples/output.csv")

In [ ]:
show_text_file("examples/output.csv")


除了逗号，也可以指定其他分隔符。  
下面使用 `sys.stdout`，表示不写入文件，而是直接把结果打印出来。


In [ ]:
import sys
data.to_csv(sys.stdout, sep="|")

写出数据时，可以通过 `na_rep` 指定缺失值在文件中的表示方式。

In [ ]:
data.to_csv(sys.stdout, na_rep="NULL")

如果不希望写出行索引或列名，可以设置 `index=False` 或 `header=False`。

In [ ]:
data.to_csv(sys.stdout, index=False, header=False)

还可以只写出部分列，并指定列的输出顺序。

In [ ]:
data.to_csv(sys.stdout, index=False, columns=["a", "b", "c"])

### <font color='darkorange'><b>动手练习 2</b></font>

#### 题目
请将 `data` 写出为一个新的 CSV 文件 `examples/practice_output.csv`，要求：

1. 不写出行索引
2. 只写出 `a`、`b`、`message` 三列
3. 缺失值用字符串 `"MISSING"` 表示
4. 写出后用 `show_text_file()` 查看文件内容

#### 你的答案
请在下方代码单元中完成练习。


In [ ]:
# Write your code here



#### 参考答案

<details>
<summary>点击查看示例代码</summary>

```python
data.to_csv(
    "examples/practice_output.csv",
    index=False,
    columns=["a", "b", "message"],
    na_rep="MISSING"
)

show_text_file("examples/practice_output.csv")
```

</details>


### 2.8 使用 Python `csv` 模块处理分隔符格式

大多数表格型文本数据都可以直接用 Pandas 读取。  
但有些非常特殊的分隔符文件，可能需要借助 Python 内置的 `csv` 模块进行手动处理。


In [ ]:
show_text_file("examples/ex7.csv")


对任意单字符分隔符文件，可以把已打开的文件对象传给 `csv.reader()`。

In [ ]:
f = open("examples/ex7.csv", encoding="utf-8")
reader = csv.reader(f)


对 `reader` 进行迭代，会逐行返回列表，并自动处理引号。

In [ ]:
for line in reader:
    print(line)
f.close()

为了把读取结果整理成字典，先将所有行读入一个列表。

In [ ]:
with open("examples/ex7.csv", encoding="utf-8") as f:
    lines = list(csv.reader(f))


然后将第一行作为表头，其余行作为数据。

In [ ]:
header, values = lines[0], lines[1:]

最后使用字典推导式和 `zip(*values)` 将“按行组织”的数据转成“按列组织”的数据。

In [ ]:
data_dict = {h: v for h, v in zip(header, zip(*values))}
data_dict

### <font color='limegreen'><b>技巧与提示</b></font>

多数情况下，优先使用 Pandas 的 `read_csv()`。  
只有在文件格式特别不规则，或者需要非常细粒度控制时，才考虑使用 Python 内置的 `csv` 模块手动解析。


## 3 读取和写入 Microsoft Excel 文件

Pandas 可以通过 `ExcelFile` 或 `read_excel()` 读取 Excel 文件。  
对于 `.xlsx` 文件，通常需要安装 `openpyxl`。

当一个 Excel 文件中包含多个工作表时，先创建 `ExcelFile` 对象会更方便。


In [ ]:
xlsx = pd.ExcelFile("examples/ex1.xlsx")

可以使用 `.sheet_names` 查看 Excel 文件中的工作表名称。

In [ ]:
xlsx.sheet_names

In [ ]:
xlsx.parse(sheet_name="Sheet1")

In [ ]:
xlsx.parse(sheet_name="Sheet1", index_col=0)

也可以直接使用 `pd.read_excel()` 读取 Excel 文件中的指定工作表。

In [ ]:
frame = pd.read_excel("examples/ex1.xlsx", sheet_name="Sheet1")
frame

如果要写出为 Excel 文件，可以使用 `ExcelWriter`。  
推荐使用 `with` 语句自动保存和关闭文件。


In [ ]:
with pd.ExcelWriter("examples/ex2.xlsx") as writer:
    frame.to_excel(writer, "Sheet1", index=False)


如果只写出一个工作表，也可以直接把路径传给 `to_excel()`。

In [ ]:
frame.to_excel("examples/ex2.xlsx")

### <font color='darkorange'><b>动手练习 3</b></font>

#### 题目
请读取 `examples/ex1.xlsx` 中的 `Sheet1`，然后将其前两列写入新的 Excel 文件 `examples/practice_excel.xlsx`。

#### 你的答案
请在下方代码单元中完成练习。


In [ ]:
# Write your code here



#### 参考答案

<details>
<summary>点击查看示例代码</summary>

```python
excel_data = pd.read_excel("examples/ex1.xlsx", sheet_name="Sheet1")
excel_data[["a", "b"]].to_excel("examples/practice_excel.xlsx", index=False)
```

</details>


In [ ]:
# 安全地删除文件
Path("examples/ex2.xlsx").unlink(missing_ok=True)

## 课堂小结

在本讲中，我们学习了 Pandas 中常用的数据读取与写入方法。

| 知识点 | 主要内容 |
|---|---|
| **读取 CSV** | `pd.read_csv()` |
| **读取分隔符文本** | `pd.read_table()`、`sep` 参数 |
| **无表头文件** | `header=None`、`names` |
| **设置索引列** | `index_col` |
| **跳过无关行** | `skiprows` |
| **缺失值识别** | `na_values`、`keep_default_na` |
| **逐块读取** | `chunksize` |
| **写出 CSV** | `DataFrame.to_csv()` |
| **手动解析 CSV** | Python 内置 `csv` 模块 |
| **读取 Excel** | `pd.ExcelFile()`、`pd.read_excel()` |
| **写出 Excel** | `DataFrame.to_excel()`、`ExcelWriter` |


### <font color='cornflowerblue'><b>思考题 2</b></font>

在实际项目中，读取一个陌生 CSV 文件时，你会先检查哪些问题？

#### 参考答案

<details>
<summary>点击查看解释</summary>

通常可以先检查：

1. 文件编码是否正确，例如 UTF-8、GBK
2. 是否有表头
3. 分隔符是什么，例如逗号、制表符、空格
4. 是否存在说明行、注释行或页脚
5. 缺失值用什么标记表示
6. 哪些列应该作为索引
7. 文件是否太大，是否需要逐块读取

</details>


<div class="alert alert-success">

**进一步学习资源**

- Pandas 官方文档：https://pandas.pydata.org/docs/
- Pandas 官方教程：https://pandas.pydata.org/docs/getting_started/index.html
- Pandas 速查表（Cheat Sheet）：https://pandas.pydata.org/Pandas_Cheat_Sheet.pdf

</div>

---

*本节课到此结束，感谢大家的学习！如有疑问，请随时提问。*